In [25]:
%load_ext autoreload
%autoreload 2
import pandas as pd
import torch
from datasets import Dataset
from dotenv import load_dotenv
from accelerate import Accelerator
from PromptTemplate import PromptTemplate
from HuggingFaceModel import HuggingFaceModel
from TrainStrategy import TrainStrategy
from constant import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
load_dotenv()
accelerator = Accelerator()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [20]:
prompt_template = PromptTemplate(
    name="Manually Crafted",
    definition="You are Code Expert trained to detect Self-Admitted Technical Debt (SATD) in Java test code comments. SATD occurs when developers explicitly acknowledge that the current implementation is suboptimal, requires improvement, or contains technical compromises. These comments often include markers (e.g., TODO, FIXME), indicate unresolved issues, temporary fixes (e.g., workarounds, hacks), performance concerns, deprecated API usage, unsupported features, poor design, skipped tests, or unknown reasons. However, do not classify comments that merely describe expected behavior, actions, instructions or simply issue references, unless there is additional information indicating the need for future improvement. These comments often appear as imperative sentence structures (e.g., check argument, should not match), vague single-word description(e.g., clean up, retry, fail).",
    instruction="Classify by labelling it as 'yes' if the comment include a strong indication of Self-Admitted Technical Debt otherwise label it as 'no' without label, do not return reason. Do not provide a reason for the classification.",
    n_shot_template="""
    Comment: {{ text }}
    """,
    n_shot_answer_template= """
    {% if cot -%}
    Answer: {{ cot }} The answer is **{{ label }}**.
    {% endif -%}
    """,
    line_m_before=3,
    line_n_after=3
)

In [28]:
for model_name in ['google/flan-t5-small']:
    flan_t5_detection_model = HuggingFaceModel('detect', model_name, {'yes', 'no'}, DEFAULT_DETECTION_CLASS)
    flan_t5_detection_model.fit(detect_n_shot_dataset)
    flan_t5_detection_model.predict(detect_test_dataset.select(range(1)), DETECT_DATASET_NAME,  prompt_template,TrainStrategy.N_SHOT_TOP,2, verbose=True)


detect with flan-t5-small
Prompt:
 You are Code Expert trained to detect Self-Admitted Technical Debt (SATD) in Java test code comments. SATD occurs when developers explicitly acknowledge that the current implementation is suboptimal, requires improvement, or contains technical compromises. These comments often include markers (e.g., TODO, FIXME), indicate unresolved issues, temporary fixes (e.g., workarounds, hacks), performance concerns, deprecated API usage, unsupported features, poor design, skipped tests, or unknown reasons. However, do not classify comments that merely describe expected behavior, actions, instructions or simply issue references, unless there is additional information indicating the need for future improvement. These comments often appear as imperative sentence structures (e.g., check argument, should not match), vague single-word description(e.g., clean up, retry, fail). Classify by labelling it as 'yes' if the comment include a strong indication of Self-Admitted

# Flan T5 Models

In [ ]:
for model_name in ['google/flan-t5-small', 'google/flan-t5-base', 'google/flan-t5-large', 'google/flan-t5-xl', 'google/flan-t5-xxl']:
    flan_t5_detection_model = HuggingFaceModel('detect', model_name, {'yes', 'no'}, DEFAULT_DETECTION_CLASS)
    flan_t5_detection_model.fit(detect_n_shot_dataset)
    flan_t5_detection_model.predict(detect_test_dataset, DETECT_DATASET_NAME,  prompt_template,TrainStrategy.N_SHOT_TOP,0, verbose=False)


detect with flan-t5-small


Token indices sequence length is longer than the specified maximum sequence length for this model (933 > 512). Running this sequence through the model will result in indexing errors


Test Result:
              precision    recall  f1-score   support

          no      0.999     0.571     0.727      6855
         yes      0.035     0.955     0.068       112

    accuracy                          0.577      6967
   macro avg      0.517     0.763     0.397      6967
weighted avg      0.983     0.577     0.716      6967



Loading checkpoint shards: 100%|██████████| 2/2 [00:08<00:00,  4.29s/it]
Some parameters are on the meta device because they were offloaded to the cpu.


detect with flan-t5-xl


Token indices sequence length is longer than the specified maximum sequence length for this model (933 > 512). Running this sequence through the model will result in indexing errors
